Supervised Fine Tuning on Open Source LLM

In [1]:
import torch

print("CUDA available:", torch.cuda.is_available())
print("GPU name:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "No GPU")

CUDA available: True
GPU name: NVIDIA GeForce RTX 4050 Laptop GPU


In [2]:
import sys
import torch

print("Python:", sys.executable)
print("Torch version:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
print("Torch CUDA version:", torch.version.cuda)
print("Device count:", torch.cuda.device_count())

Python: c:\Users\samar\RLHF_Impl\venv\Scripts\python.exe
Torch version: 2.10.0+cu126
CUDA available: True
Torch CUDA version: 12.6
Device count: 1


In [3]:
import sys
print(sys.executable)

c:\Users\samar\RLHF_Impl\venv\Scripts\python.exe


In [4]:
import os
import math
import random
from dataclasses import dataclass
from typing import Dict, List

import torch
from torch import nn
from torch.utils.data import Dataset, DataLoader
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForCausalLM, get_linear_schedule_with_warmup

MODEL_NAME = "meta-llama/Llama-3.2-1B"
#MODEL_NAME = "Qwen/Qwen2.5-1.5B"
DATASET_NAME = "tatsu-lab/alpaca"
MAX_LENGTH = 128
TRAIN_BATCH_SIZE = 4
EVAL_BATCH_SIZE = 4
GRAD_ACCUM_STEPS = 1
NUM_EPOCHS = 2
LEARNING_RATE = 2e-5
WEIGHT_DECAY = 0.01
WARMUP_RATIO = 0.03
TRAIN_VAL_SPLIT = 0.1
SEED = 42
LOG_EVERY = 10
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
OUTPUT_DIR = "./sft/checkpoints/llama32_1b_alpaca_sft"

USE_BF16 = torch.cuda.is_available() and torch.cuda.is_bf16_supported()
USE_FP16 = torch.cuda.is_available() and not USE_BF16

Prompt formatting

In [5]:
def format_prompt(example: Dict[str, str]) -> str:
    instruction = (example.get("instruction") or "").strip()
    input_text = (example.get("input") or "").strip()

    if input_text:
        return (
            "Below is an instruction that describes a task, paired with an input that provides further context.\n\n"
            f"### Instruction:\n{instruction}\n\n"
            f"### Input:\n{input_text}\n\n"
            "### Response:\n"
        )
    else:
        return (
            "Below is an instruction that describes a task.\n\n"
            f"### Instruction:\n{instruction}\n\n"
            "### Response:\n"
        )
    
def set_seed(seed: int):
    random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

Dataset:

    This class creates:
      input_ids :- prompt + response + eos
      attention_mask :- standard mask
      labels :- -100 for prompt tokens, actual ids for response tokens

In [ ]:
class AlpacaSFTDataset(Dataset):

    def __init__(self, hf_dataset, tokenizer, max_length: int):
        self.dataset = hf_dataset
        self.tokenizer = tokenizer
        self.max_length = max_length

    def __len__(self):
        return len(self.dataset)

    def __getitem__(self, idx):
        ex = self.dataset[idx]

        prompt = format_prompt(ex)
        response = (ex.get("output") or "").strip()

        # Add EOS to the target completion
        full_text = prompt + response + self.tokenizer.eos_token

        # Tokenizing full sequence
        full_enc = self.tokenizer(
            full_text,
            truncation=True,
            max_length=self.max_length,
            add_special_tokens=True,
            return_attention_mask=True,
        )

        # Tokenize prompt alone to find label masking boundary
        prompt_enc = self.tokenizer(
            prompt,
            truncation=True,
            max_length=self.max_length,
            add_special_tokens=True,
            return_attention_mask=False,
        )

        input_ids = full_enc["input_ids"]
        attention_mask = full_enc["attention_mask"]

        prompt_len = len(prompt_enc["input_ids"])
        # Mask prompt tokens and only learn on response tokens
        labels = [-100] * len(input_ids)
        for i in range(prompt_len, len(input_ids)):
            labels[i] = input_ids[i]

        return {
            "input_ids": input_ids,
            "attention_mask": attention_mask,
            "labels": labels,
        }

Collator

In [ ]:

@dataclass
class SFTCollator:
    tokenizer: AutoTokenizer

    def __call__(self, batch: List[Dict[str, List[int]]]) -> Dict[str, torch.Tensor]:
        pad_id = self.tokenizer.pad_token_id
        if pad_id is None:
            raise ValueError("Tokenizer pad_token_id is None. Set tokenizer.pad_token first.")

        max_len = max(len(x["input_ids"]) for x in batch)

        batch_input_ids = []
        batch_attention_mask = []
        batch_labels = []

        for x in batch:
            seq_len = len(x["input_ids"])
            pad_len = max_len - seq_len

            batch_input_ids.append(x["input_ids"] + [pad_id] * pad_len)
            batch_attention_mask.append(x["attention_mask"] + [0] * pad_len)
            batch_labels.append(x["labels"] + [-100] * pad_len)

        return {
            "input_ids": torch.tensor(batch_input_ids, dtype=torch.long),
            "attention_mask": torch.tensor(batch_attention_mask, dtype=torch.long),
            "labels": torch.tensor(batch_labels, dtype=torch.long),
        }



Evaluation and Metrics


In [ ]:

#Causal LM predicts token t using position t-1 logits.
#So compare logits[:, :-1, :] against labels[:, 1:].
#Ignore labels == -100.
@torch.no_grad()
def compute_token_accuracy(logits: torch.Tensor, labels: torch.Tensor) -> float:
    shift_logits = logits[:, :-1, :].contiguous()
    shift_labels = labels[:, 1:].contiguous()

    preds = shift_logits.argmax(dim=-1)
    mask = shift_labels != -100

    if mask.sum().item() == 0:
        return 0.0

    correct = (preds[mask] == shift_labels[mask]).sum().item()
    total = mask.sum().item()
    return correct / total


@torch.no_grad()
def evaluate(model, dataloader, device):
    model.eval()

    total_loss = 0.0
    total_acc = 0.0
    total_batches = 0

    for batch in dataloader:
        batch = {k: v.to(device) for k, v in batch.items()}

        outputs = model(
            input_ids=batch["input_ids"],
            attention_mask=batch["attention_mask"],
            labels=batch["labels"],
        )

        print(f"Sample logits:{outputs.logits[0,0,0]}")

        loss = outputs.loss
        acc = compute_token_accuracy(outputs.logits, batch["labels"])

        total_loss += loss.item()
        total_acc += acc
        total_batches += 1

    avg_loss = total_loss / max(total_batches, 1)
    avg_acc = total_acc / max(total_batches, 1)
    perplexity = math.exp(min(avg_loss, 20))  # avoid overflow

    return {
        "loss": avg_loss,
        "perplexity": perplexity,
        "token_accuracy": avg_acc,
    }


Training:

In [ ]:

def main():
    set_seed(SEED)
    os.makedirs(OUTPUT_DIR, exist_ok=True)

    print(f"Using device: {DEVICE}")
    print("Loading tokenizer...")
    tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, use_fast=True)
    #tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, trust_remote_code=True)

    # Many causal LMs don't define a pad token. For training, using EOS as PAD is practical.
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token

    print("Loading model...")
    model = AutoModelForCausalLM.from_pretrained(MODEL_NAME)
    #model = AutoModelForCausalLM.from_pretrained(MODEL_NAME, trust_remote_code=True)
    model.config.pad_token_id = tokenizer.pad_token_id
    model.to(DEVICE)

    print("Loading dataset...")
    raw_ds = load_dataset(DATASET_NAME, split="train[:1000]")

    split_ds = raw_ds.train_test_split(test_size=TRAIN_VAL_SPLIT, seed=SEED)
    train_ds_raw = split_ds["train"]
    val_ds_raw = split_ds["test"]

    print(f"Train size: {len(train_ds_raw)}")
    print(f"Val size:   {len(val_ds_raw)}")

    train_ds = AlpacaSFTDataset(train_ds_raw, tokenizer, MAX_LENGTH)
    val_ds = AlpacaSFTDataset(val_ds_raw, tokenizer, MAX_LENGTH)

    print(f"Sample dataset: {raw_ds[0]}")

    collator = SFTCollator(tokenizer)

    train_loader = DataLoader(
        train_ds,
        batch_size=TRAIN_BATCH_SIZE,
        shuffle=True,
        collate_fn=collator,
        num_workers=0,
        pin_memory=torch.cuda.is_available(),
    )

    val_loader = DataLoader(
        val_ds,
        batch_size=EVAL_BATCH_SIZE,
        shuffle=False,
        collate_fn=collator,
        num_workers=0,
        pin_memory=torch.cuda.is_available(),
    )


    optimizer = torch.optim.AdamW(
        model.parameters(),
        lr=LEARNING_RATE,
        weight_decay=WEIGHT_DECAY,
    )

    total_update_steps = math.ceil(len(train_loader) / GRAD_ACCUM_STEPS) * NUM_EPOCHS
    warmup_steps = int(total_update_steps * WARMUP_RATIO)

    scheduler = get_linear_schedule_with_warmup(
        optimizer,
        num_warmup_steps=warmup_steps,
        num_training_steps=total_update_steps,
    )

    scaler = torch.amp.GradScaler(enabled=USE_FP16)

    best_val_loss = float("inf")
    global_step = 0

    print("Starting training...")
    for epoch in range(NUM_EPOCHS):
        model.train()
        optimizer.zero_grad()

        running_loss = 0.0

        for step, batch in enumerate(train_loader):
            batch = {k: v.to(DEVICE) for k, v in batch.items()}

            with torch.autocast(
                device_type="cuda",
                dtype=torch.bfloat16 if USE_BF16 else torch.float16,
                enabled=(USE_BF16 or USE_FP16),
            ):
                outputs = model(
                    input_ids=batch["input_ids"],
                    attention_mask=batch["attention_mask"],
                    labels=batch["labels"],
                )
                loss = outputs.loss / GRAD_ACCUM_STEPS

            if USE_FP16:
                scaler.scale(loss).backward()
            else:
                loss.backward()

            running_loss += loss.item() * GRAD_ACCUM_STEPS

            if (step + 1) % GRAD_ACCUM_STEPS == 0:
                if USE_FP16:
                    scaler.unscale_(optimizer)

                torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)

                if USE_FP16:
                    scaler.step(optimizer)
                    scaler.update()
                else:
                    optimizer.step()

                scheduler.step()
                optimizer.zero_grad()
                global_step += 1

                if global_step % LOG_EVERY == 0:
                    avg_train_loss = running_loss / LOG_EVERY
                    current_lr = scheduler.get_last_lr()[0]
                    print(
                        f"Epoch {epoch + 1}/{NUM_EPOCHS} | "
                        f"Step {global_step} | "
                        f"Train Loss: {avg_train_loss:.4f} | "
                        f"LR: {current_lr:.6e}"
                    )
                    running_loss = 0.0

        # End-of-epoch evaluation
        val_metrics = evaluate(model, val_loader, DEVICE)
        print(
            f"\n[Epoch {epoch + 1}] "
            f"Val Loss: {val_metrics['loss']:.4f} | "
            f"Val PPL: {val_metrics['perplexity']:.4f} | "
            f"Val Token Acc: {val_metrics['token_accuracy']:.4f}\n"
        )

        # Save best checkpoint
        if val_metrics["loss"] < best_val_loss:
            best_val_loss = val_metrics["loss"]
            best_dir = os.path.join(OUTPUT_DIR, "best")
            os.makedirs(best_dir, exist_ok=True)

            model.save_pretrained(best_dir)
            tokenizer.save_pretrained(best_dir)

            torch.save(
                {
                    "epoch": epoch + 1,
                    "best_val_loss": best_val_loss,
                    "model_name": MODEL_NAME,
                    "max_length": MAX_LENGTH,
                },
                os.path.join(best_dir, "training_meta.pt"),
            )
            print(f"Saved best model to: {best_dir}")

    # Save final checkpoint
    final_dir = os.path.join(OUTPUT_DIR, "final")
    os.makedirs(final_dir, exist_ok=True)
    model.save_pretrained(final_dir)
    tokenizer.save_pretrained(final_dir)

    torch.save(
        {
            "epochs": NUM_EPOCHS,
            "model_name": MODEL_NAME,
            "max_length": MAX_LENGTH,
        },
        os.path.join(final_dir, "training_meta.pt"),
    )

    print(f"Training complete. Final model saved to: {final_dir}")

if __name__ == "__main__":
    main()


Using device: cuda
Loading tokenizer...
Loading model...


Loading weights:   0%|          | 0/146 [00:00<?, ?it/s]

Loading dataset...
Train size: 900
Val size:   100
Sample dataset: {'input_ids': [128000, 39314, 374, 459, 7754, 430, 16964, 264, 3465, 382, 14711, 30151, 512, 75885, 279, 7434, 315, 5933, 4221, 8863, 382, 14711, 6075, 512, 55381, 11688, 29225, 320, 45, 12852, 8, 374, 279, 5845, 369, 19002, 311, 3619, 323, 1920, 3823, 4221, 13, 1102, 5829, 39603, 5706, 11, 6500, 8198, 11, 323, 21075, 11478, 311, 24564, 323, 14532, 5439, 477, 22066, 3823, 4221, 304, 2015, 311, 8819, 7438, 323, 2804, 9256, 13, 1102, 649, 387, 1511, 311, 11388, 7537, 11, 49229, 22755, 11, 7068, 70022, 323, 26793, 11, 439, 1664, 439, 10695, 12933, 3619, 11, 14532, 323, 95340, 449, 12966, 13, 128001], 'attention_mask': [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1], 'l

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Saved best model to: ./sft/checkpoints/llama32_1b_alpaca_sft\best
Epoch 2/2 | Step 230 | Train Loss: 0.4598 | LR: 1.006865e-05
Epoch 2/2 | Step 240 | Train Loss: 0.9519 | LR: 9.610984e-06
Epoch 2/2 | Step 250 | Train Loss: 0.9102 | LR: 9.153318e-06
Epoch 2/2 | Step 260 | Train Loss: 0.8003 | LR: 8.695652e-06
Epoch 2/2 | Step 270 | Train Loss: 0.9316 | LR: 8.237986e-06
Epoch 2/2 | Step 280 | Train Loss: 0.8117 | LR: 7.780320e-06
Epoch 2/2 | Step 290 | Train Loss: 0.9079 | LR: 7.322654e-06
Epoch 2/2 | Step 300 | Train Loss: 0.8823 | LR: 6.864989e-06
Epoch 2/2 | Step 310 | Train Loss: 0.8749 | LR: 6.407323e-06
Epoch 2/2 | Step 320 | Train Loss: 0.8403 | LR: 5.949657e-06
Epoch 2/2 | Step 330 | Train Loss: 0.9168 | LR: 5.491991e-06
Epoch 2/2 | Step 340 | Train Loss: 1.1010 | LR: 5.034325e-06
Epoch 2/2 | Step 350 | Train Loss: 0.9780 | LR: 4.576659e-06
Epoch 2/2 | Step 360 | Train Loss: 0.7798 | LR: 4.118993e-06
Epoch 2/2 | Step 370 | Train Loss: 0.9416 | LR: 3.661327e-06
Epoch 2/2 | Step 38

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Training complete. Final model saved to: ./sft/checkpoints/llama32_1b_alpaca_sft\final


In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

MODEL_DIR = "./sft/checkpoints/llama32_1b_alpaca_sft/best"
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

tokenizer = AutoTokenizer.from_pretrained(MODEL_DIR)
model = AutoModelForCausalLM.from_pretrained(MODEL_DIR).to(DEVICE)
model.eval()

prompt = (
    "Below is an instruction that describes a task.\n\n"
    "### Instruction:\n"
    "Explain what RLHF is in simple terms.\n\n"
    "### Response:\n"
)

inputs = tokenizer(prompt, return_tensors="pt").to(DEVICE)

with torch.no_grad():
    output_ids = model.generate(
        **inputs,
        max_new_tokens=128,
        do_sample=True,
        temperature=0.8,
        top_p=0.9,
        pad_token_id=tokenizer.pad_token_id,
        eos_token_id=tokenizer.eos_token_id,
    )

print(tokenizer.decode(output_ids[0], skip_special_tokens=True))

Comparison of responses before and after Supervised Fine Tuning

In [ ]:
BASE_MODEL = MODEL_NAME
SFT_MODEL = "./sft/checkpoints/llama32_1b_alpaca_sft/best"

device = "cuda" if torch.cuda.is_available() else "cpu"

print("Loading tokenizer...")
tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL, trust_remote_code=True)

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

print("Loading BASE model...")
base_model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    trust_remote_code=True
).to(device)

print("Loading SFT model...")
sft_model = AutoModelForCausalLM.from_pretrained(
    SFT_MODEL,
    trust_remote_code=True
).to(device)

base_model.eval()
sft_model.eval()

Loading tokenizer...
Loading BASE model...


Loading weights:   0%|          | 0/146 [00:00<?, ?it/s]

Loading SFT model...


Loading weights:   0%|          | 0/146 [00:00<?, ?it/s]

LlamaForCausalLM(
  (model): LlamaModel(
    (embed_tokens): Embedding(128256, 2048, padding_idx=128001)
    (layers): ModuleList(
      (0-15): 16 x LlamaDecoderLayer(
        (self_attn): LlamaAttention(
          (q_proj): Linear(in_features=2048, out_features=2048, bias=False)
          (k_proj): Linear(in_features=2048, out_features=512, bias=False)
          (v_proj): Linear(in_features=2048, out_features=512, bias=False)
          (o_proj): Linear(in_features=2048, out_features=2048, bias=False)
        )
        (mlp): LlamaMLP(
          (gate_proj): Linear(in_features=2048, out_features=8192, bias=False)
          (up_proj): Linear(in_features=2048, out_features=8192, bias=False)
          (down_proj): Linear(in_features=8192, out_features=2048, bias=False)
          (act_fn): SiLUActivation()
        )
        (input_layernorm): LlamaRMSNorm((2048,), eps=1e-05)
        (post_attention_layernorm): LlamaRMSNorm((2048,), eps=1e-05)
      )
    )
    (norm): LlamaRMSNorm((2048,)

In [14]:
prompt = """
Below is an instruction that describes a task.

### Instruction:
How to be efficient?

### Response:
"""
inputs = tokenizer(prompt, return_tensors="pt").to(device)
def generate(model):
    with torch.no_grad():
        output = model.generate(
            **inputs,
            max_new_tokens=120,
            do_sample=True,
            temperature=0.7,
            top_p=0.9,
            pad_token_id=tokenizer.pad_token_id
        )
    return tokenizer.decode(output[0], skip_special_tokens=True)


print("\n===== BASE MODEL =====\n")
print(generate(base_model))

print("\n===== SFT MODEL =====\n")
print(generate(sft_model))


===== BASE MODEL =====


Below is an instruction that describes a task.

### Instruction:
How to be efficient?

### Response:
I would do it with 2 arrays of integers. One array for the first 10 values of the sequence and the other for the remaining values. The first array would be the input and the second array would be the output. Then I would do a for loop to go through the input array and use the modulo operator to determine if the value is even or odd. If the value is even, I would add it to the even array. If the value is odd, I would add it to the odd array. After the loop is complete, I would use a second for loop to go through

===== SFT MODEL =====


Below is an instruction that describes a task.

### Instruction:
How to be efficient?

### Response:
One way to be efficient is to plan ahead. By knowing what you need to accomplish and how long it will take, you can make sure you are not wasting time on unproductive activities. Another way to be efficient is to focus on the most